# AIoT Energy Anomaly Detection — end-to-end, one notebook (Kaggle/Jupyter)

Runs the whole pipeline from raw data to the manuscript numbers, then prints the results.
Nothing is duplicated here: every computation is a subprocess call into the repository's own
`scripts/` and `src/` modules, and every number is read from the JSON artefacts they write.

**Set-up on Kaggle (one-time):**
1. The GitHub repository is private, so a `git clone` will not work. Compress the project
   folder to **`lead.zip`** and upload it as a notebook Input (Add input → Upload → `lead.zip`).
2. Notebook settings → **Internet → On** (the dataset, ~58 MB, is downloaded from its public
   URL and verified against a pinned SHA-256).
3. `RUN_FAST = True` (below) skips the tuning grid and the weighted ablation, because a Kaggle
   CPU session has 4 cores versus the 12 used for the published run. Set it to `False` for the
   exact published configuration; expect roughly 1–2 h on Kaggle in either case.

In [ ]:
RUN_FAST = True            # True: skip tuning grid + weighted ablation (recommended on Kaggle)
REPO_ZIP_NAME = 'lead.zip'  # name of the uploaded input

In [ ]:
# 1. Unpack (or clone) the repository into /kaggle/working
from pathlib import Path
import zipfile, subprocess, sys, os

working = Path('/kaggle/working')
stem = REPO_ZIP_NAME[:-4]
candidates = [Path('/kaggle/input') / stem / REPO_ZIP_NAME,
              Path('/kaggle/input') / REPO_ZIP_NAME,
              working / REPO_ZIP_NAME]
zip_path = next((p for p in candidates if p.exists()), None)
repo = working / 'LEAD'

if not repo.exists():
    if zip_path is not None:
        print(f'[unzip] {zip_path}')
        zipfile.ZipFile(zip_path).extractall(working)
        if not repo.exists():   # the zip may extract files at the top level instead of a folder
            repo = working
    else:
        print('[clone] trying git clone (works only if the repository is public)')
        r = subprocess.run(['git', 'clone',
                            'https://github.com/h4yleyL33/LEAD.git', str(repo)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise SystemExit(
                'Repository not available. Compress the project folder to '
                f"{REPO_ZIP_NAME}, upload it as a Kaggle input, then re-run this cell.\n"
                + r.stderr)

os.chdir(repo)
sys.path.insert(0, str(repo))
print(f'[repo] {repo.resolve()}')
assert Path('scripts/run_all.py').exists() and Path('src/utils/config.py').exists(), \
    'Not the expected repository layout - check the zip contents.'

In [ ]:
# 2. Install the pinned dependencies (exact versions used for all reported results)
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
assert r.returncode == 0, 'pip install failed - see the pip output above.'
import numpy, pandas, sklearn, xgboost
print(f"numpy {numpy.__version__} | pandas {pandas.__version__} | "
      f"sklearn {sklearn.__version__} | xgboost {xgboost.__version__}")

In [ ]:
# 3. Fetch the LEAD1.0-small dataset (public URL, checksum-verified; ~58 MB)
import subprocess, sys
r = subprocess.run([sys.executable, 'scripts/fetch_data.py'])
assert r.returncode == 0, 'Data fetch failed - is Internet enabled for this notebook?'

In [ ]:
# 4. Run the whole pipeline: audit -> EDA -> features -> splits -> leakage tests ->
#     baselines -> models -> selection -> holdout -> comparison -> SHAP ->
#     simulation -> paper validation -> full test suite
import subprocess, sys
cmd = [sys.executable, 'scripts/run_all.py'] + (['--fast'] if RUN_FAST else [])
print('[run ]', ' '.join(cmd), flush=True)
r = subprocess.run(cmd)
assert r.returncode == 0, 'A pipeline stage failed - scroll up to its traceback.'
print('[done] all stages completed')

## Results (read from the artefacts written above)

In [ ]:
# 5. Data audit
import json
audit = json.load(open('results/metrics/data_audit.json'))
v, q, t = audit['volume'], audit['quality'], audit['target']
print(f"rows            : {v['n_rows']:,}")
print(f"buildings       : {v['n_buildings']}")
print(f"span            : {v['timestamp_min']} .. {v['timestamp_max']} ({v['span_days']:.0f} days)")
print(f"missing reading : {q['missing_meter_reading']:,} ({q['missing_meter_reading_rate']:.2%})")
print(f"anomalies       : {t['n_anomaly']:,} ({t['anomaly_rate']:.2%}, imbalance 1:{t['class_imbalance_ratio']:.1f})")

In [ ]:
# 6. Causality contract, re-executed on this machine: features at time t must not
#    change if any reading after t is modified.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_features_no_leakage.py', '-q'])

In [ ]:
# 7. Model comparison on validation (baselines + every tuned candidate)
import json
b = json.load(open('results/metrics/baselines_val.json'))
sel = json.load(open('results/metrics/model_selection.json'))
rows = [('prior',    b['results']['prior']['roc_auc'],    b['results']['prior']['pr_auc'],    b['results']['prior']['brier']),
        ('logistic', b['results']['logistic']['roc_auc'], b['results']['logistic']['pr_auc'], b['results']['logistic']['brier']),
        ('iforest',  b['results']['iforest']['roc_auc'],  b['results']['iforest']['pr_auc'],  b['results']['iforest']['brier'])]
rows += [(c['variant'], c['roc_auc'], c['pr_auc'], c['brier']) for c in sel['candidates']]
rows.sort(key=lambda r: -r[1])
print(f"{'model':<18}{'ROC-AUC':>9}{'PR-AUC':>9}{'Brier':>9}")
for m, ra, pa, br in rows:
    print(f"{m:<18}{ra:>9.4f}{pa:>9.4f}{br:>9.4f}")
print(f"\nselected: {sel['selected']} (rule: {sel['rule']})")

In [ ]:
# 8. Held-out test (30 unseen buildings, scored once with the frozen model)
import json
ht = json.load(open('results/metrics/holdout_test_final.json'))
print(f"Test set          : {ht['n']:,} rows, {ht['n_buildings']} buildings, base rate {ht['base_rate']:.2%}")
print(f"ROC-AUC           : {ht['roc_auc']:.4f} (95% CI {ht['auc_ci95_low']:.4f}-{ht['auc_ci95_high']:.4f}, {ht['n_boot_valid']} bootstrap)")
print(f"PR-AUC            : {ht['pr_auc']:.4f}")
print(f"Brier             : {ht['brier']:.5f} (ECE {ht['ece_uniform_15bin']:.5f})")
print(f"Episode recall    : {ht['event_recall']:.4f} ({ht['events_detected']}/{ht['n_events']})")
print(f"Median latency    : {ht['median_detection_latency_h']:.0f} h")
print(f"False alarms      : {ht['false_alarms_per_building_day']:.3f} per building-day")

In [ ]:
# 9. Global explanation - top SHAP features
import json, matplotlib.pyplot as plt
sh = json.load(open('results/metrics/shap_summary_final.json'))
top = sh['global_top15'][:10]
names = [f['feature'] for f in top][::-1]
vals  = [f['mean_abs_shap'] for f in top][::-1]
plt.figure(figsize=(7, 4))
plt.barh(names, vals, color='#c0504d')
plt.xlabel('mean |SHAP|')
plt.title(f"Top-10 features ({sh['method']}, n={sh['n_explained']:,})")
plt.tight_layout(); plt.show()

In [ ]:
# 10. SIMULATED geothermal transfer test - a pipeline smoke test, not plant validation
import json
sim = json.load(open('results/metrics/simulation_detection_SIMULATED.json'))
print(sim['warning'])
print()
print(f"{'equipment':<11}{'ROC-AUC':>9}{'PR-AUC':>9}{'fault_recall':>14}")
for r in sim['results']:
    print(f"{r['equipment']:<11}{r['roc_auc']:>9.4f}{r['pr_auc']:>9.4f}{r['fault_recall']:>14.4f}")
print()
print(sim['conclusion'])

**Notes on the run**

- With `RUN_FAST = True`, cells 7 (model comparison) will lack the six-candidate grid entries and
  the weighted ablation; the selected model is then the default-parameter one. Set `RUN_FAST = False`
  and re-run from cell 4 for the exact published configuration.
- With `RUN_FAST = False`, `paper/main.tex` numbers may differ from the recomputed ones because
  the tuning grid and seeds produce the published values only on the original environment.
  The full test suite run at the end of cell 4 reports exactly which headline numbers drift.
- All figures land in `results/figures/`; the dashboard is not started here (it needs the API and
  a forwarded port), but `backend/` runs unchanged on Kaggle if you want it.